# STRATA, livro de códigos aninhado: Llama-3.1-8B

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução* → **A100** com **RAM alta**. Depois *Executar tudo*. O notebook não precisa do GitHub do STRATA: o script vai embutido numa célula.

A pergunta (docs/09): o estrato 0 também vira um livro de códigos não uniforme (escala de 6 bits por bloco de 32, como o IQ4_XS, e um livro de 16 valores por tensor ajustado com a imatrix), e cada bit a mais divide cada valor do livro em dois. O preço de aninhar some?

O que ele mede, na mesma sessão e com a mesma imatrix, tudo no caminho F16 (matrizes das camadas desquantizadas para F16; o llama.cpp não arredonda as ativações nesse caminho, então os dois lados ficam iguais):
1. **Curva dos k-quants com imatrix**: Q3_K_M, Q4_K_S, Q4_K_M, Q5_K_S, Q5_K_M, Q6_K, um Q6_K com atenção e `ffn_down` em Q8_0 (para fechar o buraco de 2 bits até o Q8_0) e Q8_0; o IQ4_XS entra na fronteira usada pela régua.
2. **Livro aninhado por bloco**: níveis 0 (4,25 bits/peso), 1 (5,25) e 2 (6,25), com os filhos por tensor.
3. As variantes de `VARIANTES`: `bloco_ind` (cada largura ajustada do zero, sem aninhar: mede o preço de aninhar); opcionais `linha` (livro por linha, estilo Any-Precision puro) e `bloco_linha` (filhos por linha).
4. **Velocidade do kernel de tabela no nível 0**: o IQ4_XS já é "escala por bloco + tabela de 16 valores" e roda no llama.cpp sem patch; o `llama-bench` dele contra o Q4_K_M na A100 é a velocidade que o nível 0 teria.

**Régua (definida antes de rodar, docs/09):** bits por peso a mais que a curva dos k-quants com imatrix no mesmo KL, por nível.
- **sucesso:** os níveis 0, 1 e 2 a no máximo +0,05 bit/peso da fronteira (k-quants e IQ4_XS);
- **ganho forte:** sucesso e, além disso, algum nível ≥ 0,10 abaixo;
- **zona cinza:** algum nível entre +0,05 e +0,10 (registra, não abre kernel);
- **fracasso:** algum nível acima de +0,10, ou soma ≥ 0,34 (a dos tipos D, seção 30).

Ocupa ~90 GB de disco no pico (cada GGUF F16 é apagado depois de medido). Leva ~2 h na A100.

In [ ]:
VARIANTES = ['bloco_ind']   # além do livro por bloco com filhos por tensor; também: 'linha', 'bloco_linha'
KLD_CHUNKS = 100           # trechos na divergência KL (~65 MB de disco por trecho)
IMATRIX_CHUNKS = 125       # trechos de 512 tokens do wikitext de treino para a imatrix
MODELO_HF = "unsloth/Meta-Llama-3.1-8B"

In [ ]:
import os, re, json, shutil, subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'Sem GPU NVIDIA: troque o tipo de ambiente de execução para A100.'
GPU = gpu.stdout.strip().splitlines()[0]
N = os.cpu_count()
RAM = int(next(l for l in open('/proc/meminfo') if l.startswith('MemTotal')).split()[1]) / 1e6
DISCO = shutil.disk_usage('/content').free / 1e9
print(f'GPU: {GPU} | CPU: {N} threads | RAM {RAM:.0f} GB | disco livre {DISCO:.0f} GB')
if DISCO < 110:
    print('Aviso: pouco disco. Ponha KLD_CHUNKS menor ou VARIANTES = [].')
if RAM < 50:
    print('Aviso: pouca RAM. Ligue a RAM alta.')

In [ ]:
LLAMA_SHA = "2a667032dc788c3a071d538e0b572e45a3f12731"   # o mesmo commit dos outros notebooks; aqui sem patch nenhum
!rm -rf llama.cpp && git init -q llama.cpp && cd llama.cpp && git remote add origin https://github.com/ggml-org/llama.cpp.git && git fetch -q --depth 1 origin $LLAMA_SHA && git checkout -q FETCH_HEAD
!pip install -q -U transformers safetensors
!pip install -q -e llama.cpp/gguf-py
# o fim do log da compilação fica visível, para aparecer qualquer erro
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DGGML_NATIVE=ON 2>&1 | tail -3 && cmake --build build -j $(nproc) --target llama-quantize llama-perplexity llama-imatrix llama-bench 2>&1 | tail -15
!ls -la llama.cpp/build/bin | grep -E 'llama-(perplexity|quantize|imatrix|bench)$'

O quantizador do livro de códigos (`tests/kernel/strata_livro.py` do STRATA, cópia fiel). Usa a GPU pelo PyTorch.

In [ ]:
%%writefile /content/strata_livro.py
#!/usr/bin/env python3
"""Livro de códigos aninhado (experimento, só qualidade; docs/09): o estrato 0 também é um livro de códigos não uniforme,
em vez de herdar a grade uniforme do Q4_K_M, e cada bit a mais divide cada centróide em dois filhos (Any-Precision LLM).

Dois desenhos, os dois ajustados com a imatrix (erro = Σ imatrix_j · (x_ij − q_ij)², o mesmo critério do llama-quantize):

--modo linha (Any-Precision / SqueezeLLM): cada linha da matriz tem o seu livro. O livro de --semente bits sai de k-means
    ponderado 1D na linha; cada nível seguinte divide cada célula em duas com 2-médias dentro da célula (a célula do pai
    fica fixa, então o código do nível k é 2·código do nível k − 1 + filho). Custo na RAM do nível de b bits: b bits por peso
    + uma tabela de 2^b valores F16 por linha (16·2^b/colunas bits por peso).
--modo bloco (híbrido estilo IQ4_XS): cada bloco de 32 pesos tem uma escala (6 bits com sinal × um F16 por 256 pesos,
    0,25 bit/peso, a conta do IQ4_XS) e o tensor tem um livro de 2^b valores (custo desprezível). O livro do nível 0 e as
    escalas saem de rodadas alternadas (escala por bloco com o livro fixo, Lloyd do livro com as escalas fixas), partindo
    da tabela do IQ4_NL; os níveis de cima dividem cada valor do livro em dois, por tensor (--filhos tensor) ou por linha
    (--filhos linha, tabela F16 por linha), com as escalas do nível 0 fixas.
--independente: sem aninhar; cada nível é ajustado do zero com 2^b valores (mede quanto o aninhamento custa).
--modo copia: só desquantiza as matrizes do --gguf para F16 (k-quants, IQ4_XS, Q8_0 no mesmo caminho de ativações).

Sem kernel: os pesos de cada nível vão para um GGUF em F16, que o llama-perplexity lê direto, para medir a KL. Todas as
matrizes das camadas usam a mesma largura (sem a mistura do Q4_K_M, que daria mais bits às sensíveis); o resto do GGUF
(embedding, normas, saída) vem do --gguf.

    python tests/kernel/strata_livro.py --bf16 BF16.gguf --gguf Q4_K_M.gguf --imatrix imatrix.gguf \\
        --modo bloco --bits 4 5 6 --out n0.gguf n1.gguf n2.gguf
"""
import argparse, json, os, sys, time

import numpy as np
import torch

GGUF_NAMES = ("attn_q", "attn_k", "attn_v", "attn_output", "ffn_gate", "ffn_up", "ffn_down")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IQ4NL = np.array([-127, -104, -83, -65, -49, -35, -22, -10, 1, 13, 25, 38, 53, 69, 89, 113], np.float64) / 127


def ler_imatrix(path, gguf):
    """{nome do tensor: importância por coluna}, normalizada pelas contagens como o llama-quantize faz."""
    r = gguf.GGUFReader(path)
    t = {x.name: np.array(x.data, dtype=np.float32).reshape(-1) for x in r.tensors}
    out = {}
    for nome, s in t.items():
        if nome.endswith(".in_sum2"):
            base = nome[: -len(".in_sum2")]
            c = t[base + ".counts"]
            out[base] = s / c[0] if c[0] > 0 else np.ones_like(s)
    return out


# ---------------------------------------------------------------- modo linha: k-means 1D por linha, em lote

def media(w, x, c):
    return torch.where(w > 0, x / w.clamp_min(1e-300), c)


class Linhas:
    """Um lote de linhas ordenadas, com somas acumuladas por linha: a soma de qualquer intervalo da ordem sai em O(1), então
    cada rodada de Lloyd custa O(linhas × centróides), não O(pesos)."""

    def __init__(self, X, W):
        self.R, self.n = X.shape
        self.xs, self.perm = torch.sort(X, dim=1, stable=True)
        ws = torch.gather(W, 1, self.perm)
        z = torch.zeros(self.R, 1, dtype=X.dtype, device=X.device)
        self.CW = torch.cat([z, ws.cumsum(1)], 1)
        self.CX = torch.cat([z, (ws * self.xs).cumsum(1)], 1)
        self.CX2 = torch.cat([z, (ws * self.xs * self.xs).cumsum(1)], 1)
        self.ini = torch.zeros(self.R, 1, dtype=torch.long, device=X.device)
        self.fim = torch.full((self.R, 1), self.n, dtype=torch.long, device=X.device)

    def pos(self, b):
        """Índice do primeiro peso ≥ b em cada linha; b: (R, m)."""
        return torch.searchsorted(self.xs, b.contiguous())

    def somas(self, S, E):
        return self.CW.gather(1, E) - self.CW.gather(1, S), self.CX.gather(1, E) - self.CX.gather(1, S)

    def erro(self, C, S, E):
        """Σ w (x − c)² e Σ w x² do lote."""
        w, x = self.somas(S, E)
        x2 = self.CX2.gather(1, E) - self.CX2.gather(1, S)
        return float((x2 - 2 * C * x + C * C * w).sum()), float(self.CX2[:, -1].sum())

    def cortes(self, C):
        b = self.pos((C[:, :-1] + C[:, 1:]) / 2)
        return torch.cat([self.ini, b], 1), torch.cat([b, self.fim], 1)

    def kmeans(self, K, iters=30):
        """Livro de K valores por linha, partindo dos quantis ponderados. Devolve C, S, E (centróides e intervalos [S, E)
        de cada um na ordem)."""
        alvo = self.CW[:, -1:] * (torch.arange(K, dtype=self.xs.dtype, device=self.xs.device) + 0.5) / K
        i = (torch.searchsorted(self.CW.contiguous(), alvo.contiguous()) - 1).clamp(0, self.n - 1)
        C = self.xs.gather(1, i)
        for _ in range(iters):
            S, E = self.cortes(C)
            C = torch.sort(media(*self.somas(S, E), C), dim=1).values
        S, E = self.cortes(C)
        return media(*self.somas(S, E), C), S, E

    def divide(self, C, S, E, iters=12):
        """Cada célula [S, E) vira duas, com 2-médias dentro dela; devolve o livro de 2K valores (filhos intercalados)."""
        t = C.clone()
        for _ in range(iters + 1):
            m = torch.minimum(torch.maximum(self.pos(t), S), E)
            m0 = media(*self.somas(S, m), C)
            m1 = media(*self.somas(m, E), C)
            t = (m0 + m1) / 2
        R = self.R
        return (torch.stack([m0, m1], 2).reshape(R, -1), torch.stack([S, m], 2).reshape(R, -1),
                torch.stack([m, E], 2).reshape(R, -1))

    def monta(self, C, S, E):
        C = C.half().double()                                          # a tabela é guardada em F16
        v = torch.repeat_interleave(C.reshape(-1), (E - S).reshape(-1)).reshape(self.R, self.n)
        return torch.empty_like(v).scatter_(1, self.perm, v), C


def quantiza_linha(X, W, bits, semente, independente, lote):
    """Devolve [pesos do nível de cada largura em bits] e [Σ w e², Σ w x²] por nível."""
    outs = [torch.empty_like(X) for _ in bits]
    err = np.zeros((len(bits), 2))
    for a in range(0, X.shape[0], lote):
        L = Linhas(X[a:a + lote], W[a:a + lote])
        if independente:
            for i, b in enumerate(bits):
                C, S, E = L.kmeans(2 ** b)
                outs[i][a:a + L.R], Cq = L.monta(C, S, E)
                err[i] += L.erro(Cq, S, E)
            continue
        C, S, E = L.kmeans(2 ** semente)
        b = semente
        for i, alvo in enumerate(bits):
            while b < alvo:
                C, S, E = L.divide(C, S, E)
                b += 1
            outs[i][a:a + L.R], Cq = L.monta(C, S, E)
            err[i] += L.erro(Cq, S, E)
    return outs, err


# ---------------------------------------------------------------- modo bloco: escala por bloco + livro por tensor

def mais_perto(u, C):
    """Código do valor mais perto de u num livro crescente C (1D)."""
    return torch.bucketize(u, (C[:-1] + C[1:]) / 2)


def escalas(x, w, C, nf=5):
    """Escala por bloco (x, w: (blocos, 32)) que minimiza Σ w (x − s·C[q])², com q o mais perto de x/s: candidatos que
    levam o maior |x| do bloco a cada ponta do livro (os dois sinais), em ±12%, cada um refinado por mínimos quadrados."""
    j = x.abs().argmax(1, keepdim=True)
    xm = x.gather(1, j)
    xm = torch.where(xm == 0, torch.full_like(xm, 1e-30), xm)
    melhor_s = torch.ones_like(xm)
    melhor_e = torch.full_like(xm, float("inf"))
    for ponta in (C[0], C[-1]):
        for f in np.linspace(0.88, 1.12, nf):
            s = xm / (ponta * f)
            c = C[mais_perto(x / s, C)]
            den = (w * c * c).sum(1, keepdim=True)
            s = torch.where(den > 0, (w * x * c).sum(1, keepdim=True) / den.clamp_min(1e-300), s)
            s = torch.where(s == 0, torch.full_like(s, 1e-30), s)
            e = (w * (x - s * C[mais_perto(x / s, C)]) ** 2).sum(1, keepdim=True)
            m = e < melhor_e
            melhor_e = torch.where(m, e, melhor_e)
            melhor_s = torch.where(m, s, melhor_s)
    return melhor_s


def quantiza_escalas(s):
    """Escala de 6 bits com sinal (−32 … 31, sem o 0) × um F16 por 8 blocos (256 pesos), como no IQ4_XS."""
    sb = s.reshape(-1, 8)
    d = (sb.abs().amax(1, keepdim=True) / 31).half().double().clamp_min(1e-30)
    l = torch.round(sb / d).clamp(-32, 31)
    l = torch.where(l == 0, torch.where(sb < 0, -1.0, 1.0).to(l.dtype), l)
    return (l * d).reshape(-1, 1)


def lloyd_tensor(u, w, C, iters=8):
    K = len(C)
    for _ in range(iters):
        q = mais_perto(u, C).reshape(-1)
        sw = torch.bincount(q, w.reshape(-1), K)
        su = torch.bincount(q, (w * u).reshape(-1), K)
        C = torch.sort(media(sw, su, C)).values
    return C


def divide_grupos(u, w, C, g, G, iters=12):
    """Cada valor C[g] (um por grupo g: o código, ou linha·K + código) vira dois filhos por 2-médias dentro da célula;
    devolve o livro dos filhos (G, 2) e o filho de cada peso."""
    gu, gw, uu = g.reshape(-1), w.reshape(-1), u.reshape(-1)
    t = C.reshape(-1).clone()
    for _ in range(iters + 1):
        f = uu >= t[gu]
        m1 = media(torch.bincount(gu, gw * f, G), torch.bincount(gu, gw * uu * f, G), t)
        m0 = media(torch.bincount(gu, gw * ~f, G), torch.bincount(gu, gw * uu * ~f, G), t)
        t = (m0 + m1) / 2
    return torch.stack([m0, m1], 1), f.reshape(u.shape).long()


def quantiza_bloco(X, W, bits, filhos, independente, rodadas=3):
    R, n = X.shape
    x, w = X.reshape(-1, 32), W.reshape(-1, 32)
    outs = []

    x32, w32 = x.float(), w.float()                                   # a busca das escalas em F32 (4× mais rápida)

    def ajusta(C):
        for r in range(rodadas):
            s = escalas(x32, w32, C.float()).double()
            if r == rodadas - 1:
                s = quantiza_escalas(s)
            C = lloyd_tensor(x / s, w * s * s, C)
        return s, C

    def erros():
        return np.array([[float((W * (X - v) ** 2).sum()), float((W * X * X).sum())] for v in outs])

    iq4 = torch.tensor(IQ4NL, dtype=X.dtype, device=X.device)
    if independente:
        for b in bits:
            C0 = torch.tensor(np.interp(np.linspace(0, 15, 2 ** b), np.arange(16), IQ4NL), dtype=X.dtype, device=X.device)
            s, C = ajusta(C0)
            outs.append((s * C[mais_perto(x / s, C)]).reshape(R, n))
        return outs, erros()
    assert bits[0] == 4, "o modo bloco aninhado parte de 4 bits"
    s, C = ajusta(iq4)
    u, wu = x / s, w * s * s
    q = mais_perto(u, C)
    outs.append((s * C[q]).reshape(R, n))
    K = 16
    linha = (torch.arange(len(x), device=X.device) // (n // 32))[:, None]
    for b in bits[1:]:
        if filhos == "linha":
            g, G = linha * K + q, R * K
            Ccur = C.reshape(-1) if C.ndim > 1 else C.repeat(R)
        else:
            g, G, Ccur = q, K, C
        F, f = divide_grupos(u, wu, Ccur, g, G)
        q, K = 2 * q + f, 2 * K
        if filhos == "linha":
            C = F.reshape(R, K).half().double()
            outs.append((s * C[linha, q]).reshape(R, n))
        else:
            C = F.reshape(K)
            outs.append((s * C[q]).reshape(R, n))
    return outs, erros()


# ---------------------------------------------------------------- GGUF

def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--bf16", required=True)
    ap.add_argument("--gguf", required=True, help="GGUF de partida (metadados e tensores fora das camadas)")
    ap.add_argument("--imatrix")
    ap.add_argument("--modo", choices=["linha", "bloco", "copia"], required=True)
    ap.add_argument("--bits", type=int, nargs="+", default=[4, 5, 6], help="largura de cada nível gravado")
    ap.add_argument("--semente", type=int, help="modo linha: bits do primeiro k-means (padrão: o primeiro de --bits)")
    ap.add_argument("--filhos", choices=["tensor", "linha"], default="tensor", help="modo bloco: livro dos níveis de cima")
    ap.add_argument("--independente", action="store_true", help="cada nível ajustado do zero (sem aninhar)")
    ap.add_argument("--camadas", type=int, help="só as primeiras N camadas (triagem rápida por erro; as outras saem do --gguf)")
    ap.add_argument("--lote", type=int, default=1024, help="modo linha: linhas por lote")
    ap.add_argument("--out", nargs="*", default=[], help="um GGUF F16 por nível (nenhum = só o erro ponderado)")
    ap.add_argument("--json", help="grava o resumo (bits por peso e erro ponderado por nível)")
    args = ap.parse_args()
    if args.modo == "copia":
        args.bits = [0]
    semente = args.semente or args.bits[0]

    import gguf
    imat = ler_imatrix(args.imatrix, gguf) if args.imatrix else {}
    bf = {t.name: t for t in gguf.GGUFReader(args.bf16).tensors}
    reader = gguf.GGUFReader(args.gguf)
    arch = reader.fields[gguf.Keys.General.ARCHITECTURE].contents()
    writers = []
    for out in args.out:
        wr = gguf.GGUFWriter(out, arch)
        for field in reader.fields.values():
            if field.name == gguf.Keys.General.ARCHITECTURE or field.name.startswith("GGUF."):
                continue
            vt = field.types[0]
            wr.add_key_value(field.name, field.contents(), vt, sub_type=field.types[-1] if vt == gguf.GGUFValueType.ARRAY else None)
        writers.append(wr)

    def e_alvo(t):
        p = t.name.split(".")
        return (len(p) == 4 and p[0] == "blk" and p[2] in GGUF_NAMES and p[3] == "weight"
                and (args.camadas is None or int(p[1]) < args.camadas))

    alvos = [e_alvo(t) for t in reader.tensors]
    for t, alvo in zip(reader.tensors, alvos):
        for wr in writers:
            if alvo:
                rows = int(t.shape[1])
                ncols = int(t.shape[0])
                wr.add_tensor_info(t.name, (rows, ncols), np.dtype(np.float16), rows * ncols * 2, raw_dtype=gguf.GGMLQuantizationType.F16)
            else:
                wr.add_tensor_info(t.name, t.data.shape, t.data.dtype, t.data.nbytes, raw_dtype=t.tensor_type)
    for wr in writers:
        wr.write_header_to_file()
        wr.write_kv_data_to_file()
        wr.write_ti_data_to_file()

    n_pesos, bits_tab, err = 0, np.zeros(len(args.bits)), np.zeros((len(args.bits), 2))
    bytes_gguf, t0 = 0, time.time()
    for t, alvo in zip(reader.tensors, alvos):
        if not alvo:
            for wr in writers:
                wr.write_tensor_data(np.array(t.data))
            continue
        rows, ncols = int(t.shape[1]), int(t.shape[0])
        n_pesos += rows * ncols
        bytes_gguf += int(t.n_bytes)
        X = (np.array(bf[t.name].data).view(np.uint16).astype(np.uint32) << 16).view(np.float32).reshape(rows, ncols)
        X = torch.from_numpy(X).to(DEV, torch.float64)
        qw = imat.get(t.name)
        W = torch.from_numpy(np.ones(ncols) if qw is None else qw.astype(np.float64)).to(DEV).expand(rows, ncols).contiguous()
        if args.modo == "copia":
            v = gguf.quants.dequantize(np.array(t.data), t.tensor_type).astype(np.float64).reshape(rows, ncols)
            vs = [torch.from_numpy(v).to(DEV)]
            e = np.array([[float((W * (X - vs[0]) ** 2).sum()), float((W * X * X).sum())]])
        elif args.modo == "linha":
            vs, e = quantiza_linha(X, W, args.bits, semente, args.independente, args.lote)
            for i, b in enumerate(args.bits):
                bits_tab[i] += rows * 16 * 2 ** b
        else:
            vs, e = quantiza_bloco(X, W, args.bits, args.filhos, args.independente)
            for i, b in enumerate(args.bits):
                bits_tab[i] += rows * ncols * 0.25 + (rows if args.filhos == "linha" and i > 0 else 1) * 16 * 2 ** b
        err += e
        for wr, v in zip(writers, vs):
            wr.write_tensor_data(v.to(torch.float16).cpu().numpy())
        del X, W, vs
        print(f"  {t.name} {rows}×{ncols}: " + " ".join(f"{100 * a / b_:.3f}%" for a, b_ in e) + f"  ({time.time() - t0:.0f} s)",
              file=sys.stderr, flush=True)
    for wr in writers:
        wr.close()
    res = {"modo": args.modo, "bits": args.bits, "semente": semente, "filhos": args.filhos, "independente": args.independente,
           "n_pesos": n_pesos, "niveis": []}
    for i, b in enumerate(args.bits):
        bpw = bytes_gguf * 8 / n_pesos if args.modo == "copia" else b + bits_tab[i] / n_pesos
        rel = err[i, 0] / err[i, 1]
        res["niveis"].append({"bits": b, "bits_por_peso": bpw, "erro_rel": rel})
        print(f"nível {i}: {bpw:.3f} bits/peso nas matrizes das camadas, erro ponderado relativo {100 * rel:.4f}%")
    if args.json:
        json.dump(res, open(args.json, "w"), indent=1)


if __name__ == "__main__":
    main()

In [ ]:
import urllib.request, zipfile, io
try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://huggingface.co/datasets/ggml-org/ci/resolve/main/wikitext-2-raw-v1.zip').read()))
    open('/content/wiki.test.raw', 'wb').write(z.read('wikitext-2-raw/wiki.test.raw'))
    open('/content/wiki.train.raw', 'wb').write(z.read('wikitext-2-raw/wiki.train.raw'))
    TEXTO = 'wikitext-2 raw'
except Exception as e:
    print('download do wikitext raw falhou:', e)
    base = 'https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/'
    urllib.request.urlretrieve(base + 'test.txt', '/content/wiki.test.raw')
    urllib.request.urlretrieve(base + 'train.txt', '/content/wiki.train.raw')
    TEXTO = 'wikitext-2 (não-raw, cópia do PyTorch)'
print(TEXTO, os.path.getsize('/content/wiki.test.raw'), 'bytes de teste,', os.path.getsize('/content/wiki.train.raw'), 'de treino')

Modelo em BF16, imatrix, k-quants e IQ4_XS com imatrix, Q8_0 e a base da KL. Os pesos originais do Hugging Face são apagados depois da conversão.

In [ ]:
from huggingface_hub import snapshot_download
src = snapshot_download(MODELO_HF, allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
G = '/content/gguf'
os.makedirs(G, exist_ok=True)
!python llama.cpp/convert_hf_to_gguf.py {src} --outtype bf16 --outfile {G}/BF16.gguf 2>&1 | tail -1
!rm -rf {src}
!llama.cpp/build/bin/llama-imatrix -m {G}/BF16.gguf -f /content/wiki.train.raw -ngl 99 -c 512 --chunks {IMATRIX_CHUNKS} -o {G}/imatrix.gguf > {G}/imatrix.log 2>&1
!tail -2 {G}/imatrix.log
for q in ['Q3_K_M', 'Q4_K_S', 'Q4_K_M', 'Q5_K_S', 'Q5_K_M', 'Q6_K', 'IQ4_XS']:
    !llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {G}/BF16.gguf {G}/{q}_imat.gguf {q} > {G}/{q}_imat.log 2>&1
# um ponto entre o Q6_K e o Q8_0 (a curva tem 2 bits de buraco ali): Q6_K com as matrizes de atenção e a ffn_down em Q8_0
Q8MIX = ' '.join(f'--tensor-type {t}=q8_0' for t in ['attn_q', 'attn_k', 'attn_v', 'attn_output', 'ffn_down'])
!llama.cpp/build/bin/llama-quantize --imatrix {G}/imatrix.gguf {Q8MIX} {G}/BF16.gguf {G}/Q6_K_Q8mix_imat.gguf Q6_K > {G}/Q6_K_Q8mix_imat.log 2>&1
!llama.cpp/build/bin/llama-quantize {G}/BF16.gguf {G}/Q8_0.gguf Q8_0 > {G}/Q8_0.log 2>&1
!llama.cpp/build/bin/llama-perplexity -m {G}/BF16.gguf -f /content/wiki.test.raw -c 512 -ngl 99 --chunks {KLD_CHUNKS} --kl-divergence-base /content/kld_base.bin > /dev/null 2>&1
!ls -la {G}/*.gguf /content/kld_base.bin
!df -h /content | tail -1

Funções de medida. `bpw` guarda os bits por peso nas matrizes das camadas (bytes reais para os k-quants; códigos + escalas + tabelas para o livro).

In [ ]:
kld, bpw, erro, vel = {}, {}, {}, {}
LIVRO = f'python /content/strata_livro.py --bf16 {G}/BF16.gguf --imatrix {G}/imatrix.gguf'

def kl_de(path, nome):
    out = !llama.cpp/build/bin/llama-perplexity -m {path} --kl-divergence-base /content/kld_base.bin --kl-divergence -ngl 99 2>&1
    txt = '\n'.join(out)
    k_ = re.search(r'Mean +KLD: +([0-9.]+) ± +([0-9.]+)', txt)
    top = re.search(r'Same top p: +([0-9.]+) ± +([0-9.]+) %', txt)
    if not k_:
        print(f'Erro na KL de {nome}:', '\n'.join(out[-8:]))
        return
    kld[nome] = (float(k_.group(1)), float(k_.group(2)), float(top.group(1)))
    print(f'{nome}: KL ×1000 {1000 * kld[nome][0]:.2f} ± {1000 * kld[nome][1]:.2f}, mesmo 1º token {kld[nome][2]}%, {bpw.get(nome, float("nan")):.3f} bits/peso')

def livro(nome, args, niveis, base='Q4_K_M_imat'):
    outs = [f'/content/e{i}.gguf' for i in range(len(niveis))]
    r = !{LIVRO} --gguf {G}/{base}.gguf --json /content/{nome}.json {args} --out {' '.join(outs)} 2>&1
    print('\n'.join(l for l in r if l.startswith('nível') or 'Error' in l or 'Traceback' in l))
    res = json.load(open(f'/content/{nome}.json'))
    for o, n, nv in zip(outs, niveis, res['niveis']):
        if n is not None:
            bpw[f'{nome}_N{n}'] = nv['bits_por_peso']
            erro[f'{nome}_N{n}'] = nv['erro_rel']
            kl_de(o, f'{nome}_N{n}')
        !rm -f {o}

def bench(path, nome):
    # tok/s de geração (tg128) e de prompt (pp512) na GPU, 3 repetições
    out = !llama.cpp/build/bin/llama-bench -m {path} -ngl 99 -p 512 -n 128 -r 3 -o json 2>/dev/null
    try:
        r = json.loads('\n'.join(l for l in out if not l.startswith('ggml_') and not l.startswith('load')))
        vel[nome] = {('pp' if x['n_prompt'] else 'tg'): x['avg_ts'] for x in r}
        print(f'{nome}: {vel[nome]}')
    except Exception as e:
        print(f'Erro no llama-bench de {nome}:', e, '\n'.join(out[-5:]))

Velocidade na GPU dos tipos do llama.cpp sem patch: o IQ4_XS (tabela de 16 valores) contra o Q4_K_M (grade uniforme), mais Q5_K_M e Q6_K de referência.

In [ ]:
for q in ['Q4_K_M_imat', 'IQ4_XS_imat', 'Q5_K_M_imat', 'Q6_K_imat']:
    bench(f'{G}/{q}.gguf', q)

A curva: k-quants, IQ4_XS e Q8_0 com as matrizes das camadas desquantizadas para F16 (~16 GB cada). Os GGUFs quantizados vão sendo apagados; o Q4_K_M fica, porque dá os metadados e os tensores de fora das camadas ao livro.

In [ ]:
for q in ['Q3_K_M_imat', 'Q4_K_S_imat', 'Q4_K_M_imat', 'IQ4_XS_imat', 'Q5_K_S_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q6_K_Q8mix_imat', 'Q8_0']:
    r = !{LIVRO} --gguf {G}/{q}.gguf --modo copia --json /content/F16_{q}.json --out /content/e0.gguf 2>&1
    nv = json.load(open(f'/content/F16_{q}.json'))['niveis'][0]
    bpw['F16_' + q], erro['F16_' + q] = nv['bits_por_peso'], nv['erro_rel']
    kl_de('/content/e0.gguf', 'F16_' + q)
    !rm -f /content/e0.gguf
    if q != 'Q4_K_M_imat':
        !rm -f {G}/{q}.gguf
!df -h /content | tail -1

O livro aninhado por bloco (filhos por tensor), níveis 0, 1 e 2, e as variantes.

In [ ]:
livro('bloco', '--modo bloco --bits 4 5 6', [0, 1, 2])
if 'bloco_linha' in VARIANTES:
    livro('bloco_linha', '--modo bloco --bits 4 5 6 --filhos linha', [None, 1, 2])
if 'bloco_ind' in VARIANTES:
    livro('bloco_ind', '--modo bloco --bits 5 6 --independente', [1, 2])
if 'linha' in VARIANTES:
    livro('linha', '--modo linha --bits 4 5 6 --lote 4096', [0, 1, 2])

**A régua.** Para cada candidato:
- *bits/peso*: nas matrizes das camadas, com escalas e tabelas;
- *curva no mesmo tamanho*: KL dos k-quants com imatrix em F16 interpolada (log) no mesmo bits/peso;
- *bits a mais*: quanto o candidato gasta além do k-quant de mesma KL (abaixo de 0 = o livro ganha);
- *com IQ4_XS*: a mesma conta com o IQ4_XS incluído na curva.

In [ ]:
import math
from IPython.display import Markdown, display
KQ = ['Q3_K_M_imat', 'Q4_K_S_imat', 'Q4_K_M_imat', 'Q5_K_S_imat', 'Q5_K_M_imat', 'Q6_K_imat', 'Q6_K_Q8mix_imat', 'Q8_0']

def pontos(com_iq4=False):
    qs = KQ + (['IQ4_XS_imat'] if com_iq4 else [])
    p = sorted((bpw['F16_' + q], kld['F16_' + q][0]) for q in qs if 'F16_' + q in kld)
    # só a fronteira: cada ponto tem de ter KL menor que todos os de menos bits
    f = []
    for s, k in p:
        if not f or k < f[-1][1]:
            f.append((s, k))
    return f

def curva(s, pts):
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if s <= s2 or (s2, k2) == pts[-1]:
            if s < pts[0][0]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return k1 * math.exp(math.log(k2 / k1) * (s - s1) / (s2 - s1))

def bits_igual(k, pts):
    for (s1, k1), (s2, k2) in zip(pts, pts[1:]):
        if k >= k2 or (s2, k2) == pts[-1]:
            if k > pts[0][1]:
                (s1, k1), (s2, k2) = pts[0], pts[1]
            return s1 + (s2 - s1) * math.log(k / k1) / math.log(k2 / k1)

T = [f'# STRATA, livro de códigos aninhado (Llama-3.1-8B, caminho F16)', '',
     f'GPU: {GPU}. KL contra o BF16 nos {KLD_CHUNKS} primeiros trechos do {TEXTO} de teste; imatrix com {IMATRIX_CHUNKS} trechos do texto de treino. '
     'Bits por peso nas matrizes das camadas.', '',
     '| modelo | bits/peso | KL (×1000) | mesmo 1º token | erro ponderado |', '|---|---:|---:|---:|---:|']
for m in sorted(kld, key=lambda m: bpw.get(m, 99)):
    v = kld[m]
    T.append(f'| {m} | {bpw.get(m, float("nan")):.3f} | {1000 * v[0]:.2f} ± {1000 * v[1]:.2f} | {v[2]:.2f}% | {100 * erro.get(m, float("nan")):.4f}% |')
P, PI = pontos(), pontos(True)
T += ['', '## Régua (curva = k-quants com imatrix em F16)', '',
      '| candidato | bits/peso | KL ×1000 | curva no mesmo tamanho | razão | bits a mais | bits a mais (curva com IQ4_XS) |',
      '|---|---:|---:|---:|---:|---:|---:|']
extra, extra_iq = {}, {}
for m in kld:
    if m.startswith('F16_'):
        continue
    s, k = bpw[m], kld[m][0]
    extra[m], extra_iq[m] = s - bits_igual(k, P), s - bits_igual(k, PI)
    T.append(f'| {m} | {s:.3f} | {1000 * k:.2f} | {1000 * curva(s, P):.2f} | {k / curva(s, P):.2f} | {extra[m]:+.3f} | {extra_iq[m]:+.3f} |')

def veredito(nome):
    # a régua usa a fronteira com o IQ4_XS: é a mais exigente, e só muda a conta perto de 4,3 bits/peso
    e = [extra_iq.get(f'{nome}_N{n}') for n in (0, 1, 2)]
    if nome in ('bloco_ind', 'bloco_linha'):                 # o nível 0 é o mesmo do livro por bloco
        e[0] = extra_iq.get('bloco_N0')
    if None in e:
        return 'incompleto'
    if max(e) <= 0.05 and min(e) <= -0.10:
        v = 'ganho forte'
    elif max(e) <= 0.05:
        v = 'sucesso'
    elif max(e) <= 0.10 and sum(e) < 0.34:
        v = 'zona cinza'
    else:
        v = 'fracasso'
    return f'{v} (níveis 0/1/2: {e[0]:+.3f} / {e[1]:+.3f} / {e[2]:+.3f}; soma {sum(e):+.3f})'

T += ['', '## Veredito pela régua de docs/09', '']
for nome in ['bloco'] + [v for v in VARIANTES]:
    T.append(f'- **{nome}**: {veredito(nome)}')
T += ['', 'Referência (seção 30, tipos D sobre o Q4_K_M do Hub): 0 / +0,23 / +0,11 bit/peso.', '',
      '**Dominância** (menos bits e menos KL ao mesmo tempo, sem depender da interpolação):', '']
for m in kld:
    if m.startswith('F16_'):
        continue
    dom = [q.replace('F16_', '') for q in kld if q.startswith('F16_') and bpw[m] <= bpw[q] and kld[m][0] <= kld[q][0]]
    T.append(f'- {m}: ' + (', '.join(dom) if dom else 'nenhum'))
if vel:
    T += ['', '## Velocidade na GPU, llama.cpp sem patch (tok/s)', '', '| modelo | geração (tg128) | prompt (pp512) |', '|---|---:|---:|']
    for m, v in vel.items():
        T.append(f'| {m} | {v.get("tg", float("nan")):.1f} | {v.get("pp", float("nan")):.0f} |')
    a, b = 'IQ4_XS_imat', 'Q4_K_M_imat'
    if a in vel and b in vel:
        T.append(f'| IQ4_XS / Q4_K_M | {vel[a]["tg"] / vel[b]["tg"]:.2f} | {vel[a]["pp"] / vel[b]["pp"]:.2f} |')

os.makedirs('/content/res', exist_ok=True)
open('/content/res/strata_8b_livro.md', 'w').write('\n'.join(T) + '\n')
json.dump({'texto': TEXTO, 'gpu': GPU, 'kld_chunks': KLD_CHUNKS, 'imatrix_chunks': IMATRIX_CHUNKS, 'kld': kld,
           'bits_por_peso': bpw, 'erro_rel': erro, 'vel': vel, 'bits_a_mais': extra, 'bits_a_mais_com_iq4': extra_iq},
          open('/content/res/strata_8b_livro.json', 'w'), indent=1)
display(Markdown('\n'.join(T)))

Cole a tabela na thread (ou baixe os arquivos abaixo).

In [ ]:
from google.colab import files
files.download('/content/res/strata_8b_livro.md')
files.download('/content/res/strata_8b_livro.json')